# INGER | Centro de gestión inteligente
**Asistente institucional de apoyo a seguimiento, interconsultas y traslados · Demostración con datos totalmente ficticios**

La plataforma integra **planificación de herramientas mediante Groq**, ejecución de tareas en **LangGraph**, recuperación híbrida con **FAISS**, memoria conversacional persistente en **SQLite**, almacenamiento sincronizado con **Google Drive** y observabilidad mediante **LangSmith**.

**Restricciones:** no se realizan diagnósticos ni decisiones clínicas; las reglas de revisión son exclusivamente simuladas. Ninguna respuesta puede incorporar hechos que no consten en los registros de prueba. Los borradores son administrativos y no modifican las fuentes. Las API externas reciben únicamente ejemplos ficticios.

**Modo de operación:** `Entorno de ejecución → Ejecutar todas`, sin subir ZIP ni escribir claves. Groq/LangSmith leen los secretos guardados en Colab. Google puede solicitar autorización para montar Drive. Si faltan claves, el sistema informa el requisito sin imprimirlas.

**Evidencias:** ocho escenarios evaluados con aserciones, memoria SQLite separada del checkpointer, auditoría en Drive, comprobación de trazas en LangSmith y prueba de persistencia. La ejecución solo se declara correcta cuando todas las verificaciones terminan conformes.

## 01 — Preparación de componentes
Instala en el entorno de Colab las bibliotecas para orquestación, memoria, recuperación semántica y observabilidad. El código y las bases ficticias iniciales están incorporados a este notebook.

In [1]:
%pip -q install -U langgraph langgraph-checkpoint-sqlite langchain-core langchain-groq langsmith sentence-transformers faiss-cpu

## 02 — Configuración segura y automática
El notebook accede directamente a los **secretos existentes de Colab**. Deben estar guardados con los nombres `GROQ_API_KEY` y `LANGSMITH_API_KEY`, con el acceso del notebook habilitado en el panel **Secretos** (icono de llave). Se acepta `LANGCHAIN_API_KEY` como alias de LangSmith. Los valores nunca se imprimen.

`GROQ_MODEL`, `LANGSMITH_PROJECT`, `LANGSMITH_ENDPOINT` y `LANGSMITH_WORKSPACE_ID` son opcionales. Si no están guardados, se utilizan valores de configuración por defecto.

In [3]:
import os
from google.colab import userdata

def secreto_colab(*nombres, obligatorio=False):
    for nombre in nombres:
        try:
            valor = userdata.get(nombre)
            if valor and str(valor).strip():
                return str(valor).strip()
        except Exception:
            pass
    if obligatorio:
        raise RuntimeError(
            f"No se puede leer el secreto de Colab {nombres[0]}. "
            "Ábrelo en el panel Secretos (llave) y habilita 'Acceso del notebook'. "
            "El sistema no solicita ni muestra claves en pantalla."
        )
    return None

os.environ['GROQ_API_KEY'] = secreto_colab('GROQ_API_KEY', 'GROQ_KEY', 'GROQ_API', obligatorio=True)
os.environ['LANGSMITH_API_KEY'] = secreto_colab('LANGSMITH_API_KEY', 'LANGCHAIN_API_KEY', 'LANGSMITH_KEY', obligatorio=True)
os.environ['GROQ_MODEL'] = secreto_colab('GROQ_MODEL') or 'openai/gpt-oss-20b'
os.environ['LANGSMITH_PROJECT'] = secreto_colab('LANGSMITH_PROJECT') or 'INGER-Gestion-Inteligente'
os.environ['LANGSMITH_ENDPOINT'] = secreto_colab('LANGSMITH_ENDPOINT') or 'https://api.smith.langchain.com'
workspace_id = secreto_colab('LANGSMITH_WORKSPACE_ID')
if workspace_id:
    os.environ['LANGSMITH_WORKSPACE_ID'] = workspace_id
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGCHAIN_TRACING_V2'] = 'true'  # Compatibilidad con integraciones anteriores
os.environ['LANGCHAIN_PROJECT'] = os.environ['LANGSMITH_PROJECT']
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Configuración segura activada | Modelo:', os.environ['GROQ_MODEL'])
print('Proyecto de observabilidad:', os.environ['LANGSMITH_PROJECT'])
print('Groq y LangSmith: secretos detectados (valores ocultos).')

Configuración segura activada | Modelo: openai/gpt-oss-20b
Proyecto de observabilidad: INGER-Gestion-Inteligente
Groq y LangSmith: secretos detectados (valores ocultos).


## 03 — Almacenamiento de trabajo en Google Drive
Se monta Drive y se crea automáticamente `Mi unidad/INGER_Gestion_Inteligente/`. Las fuentes CSV permanecerán allí. Para evitar conflictos de acceso, las operaciones de SQLite y FAISS se realizan localmente y se sincronizan como archivos completos a Drive.

**Importante:** la autorización de Drive puede requerir hacer clic en una ventana de Google la primera vez. Colab no permite omitir ese consentimiento mediante código.

In [4]:
from google.colab import drive
from pathlib import Path
import json, csv, shutil, sqlite3, hashlib, re, time
from datetime import datetime, timezone
from IPython.display import display, Markdown

DRIVE_MOUNT = Path('/content/drive')
drive.mount(str(DRIVE_MOUNT))
BASE_DRIVE = DRIVE_MOUNT / 'MyDrive' / 'INGER_Gestion_Inteligente'
DATA_DRIVE = BASE_DRIVE / 'datos'
STATE_DRIVE = BASE_DRIVE / 'persistencia'
OUTPUT_DRIVE = BASE_DRIVE / 'reportes'
CACHE_DRIVE = BASE_DRIVE / 'indice_semantico'
for folder in (DATA_DRIVE, STATE_DRIVE, OUTPUT_DRIVE, CACHE_DRIVE):
    folder.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content/inger_motor_local')
LOCAL.mkdir(parents=True, exist_ok=True)
print('Carpeta operativa en Google Drive:', BASE_DRIVE)

Mounted at /content/drive
Carpeta operativa en Google Drive: /content/drive/MyDrive/INGER_Gestion_Inteligente


## 04 — Fuentes ficticias y reglas de demostración
La primera ejecución genera automáticamente las mismas cinco fuentes disponibles en el paquete de datos. Las ejecuciones posteriores respetan las versiones que ya estén guardadas en Drive. No es necesario cargar archivos antes de comenzar.

In [5]:
import csv, json

FUENTES_INICIALES = {'ortogeriatria_2024.csv': [['caso_id', 'fecha_alta', 'fecha_control', 'motivo_control', 'gestion_1_estado', 'gestion_1_detalle', 'gestion_2_estado', 'gestion_2_detalle'], ['CASO-001', '2024-03-05', '2024-03-20', 'Seguimiento administrativo posalta', 'realizada', 'Contacto telefónico registrado', 'realizada', 'Control confirmado'], ['CASO-003', '2024-05-10', '2024-05-25', 'Revisión de continuidad del seguimiento', 'sin_registro', 'No consta confirmación de contacto', '', ''], ['CASO-005', '2024-07-18', '2024-08-01', 'Seguimiento administrativo de alta', 'realizada', 'Se entregó información de control', 'realizada', 'Control anotado'], ['CASO-008', '2024-09-12', '2024-09-27', 'Seguimiento de coordinación institucional', 'pendiente', 'Confirmar recepción de antecedentes', '', '']], 'ortogeriatria_2025.csv': [['caso_id', 'fecha_alta', 'fecha_control', 'motivo_control', 'gestion_1_estado', 'gestion_1_detalle', 'gestion_2_estado', 'gestion_2_detalle'], ['CASO-002', '2025-01-09', '2025-01-24', 'Seguimiento administrativo posalta', 'pendiente', 'Confirmar asistencia a control', 'realizada', 'Contacto inicial registrado'], ['CASO-004', '2025-04-02', '2025-04-18', 'Gestión de continuidad de atención', 'pendiente', 'Coordinar llamada institucional', 'sin_registro', 'No consta segunda gestión'], ['CASO-006', '2025-06-17', '2025-07-03', 'Coordinación de seguimiento', 'realizada', 'Contacto realizado', 'realizada', 'Control confirmado'], ['CASO-007', '2025-08-01', '2025-08-20', 'Control programado de seguimiento', 'realizada', 'Cita coordinada', 'realizada', 'Seguimiento finalizado']], 'interconsultas_traslados.csv': [['caso_id', 'estado_interconsulta', 'estado_traslado', 'dias_espera_registrados', 'observacion_interconsulta', 'observacion_traslado'], ['CASO-001', 'resuelta', 'realizado', 2, 'Evaluación concluida', 'Coordinación registrada'], ['CASO-002', 'pendiente', 'pendiente', 9, 'Esperando respuesta de coordinación', 'Traslado pendiente de asignación'], ['CASO-003', 'sin_registro', 'sin_registro', '', 'No consta antecedente de interconsulta', 'No consta dato sobre traslado'], ['CASO-004', 'en_evaluacion', 'pendiente', 5, 'Documentación recibida; evaluación en curso', 'En espera de respuesta'], ['CASO-005', 'resuelta', 'realizado', 1, 'Interconsulta finalizada', 'Traslado coordinado'], ['CASO-006', 'resuelta', 'realizado', 3, 'Evaluación finalizada', 'Procedimiento administrativo registrado'], ['CASO-007', 'resuelta', 'no_aplica', 0, 'Se resolvió sin traslado', 'Traslado no requerido en registro'], ['CASO-008', 'pendiente', 'no_realizado', 8, 'Solicitud enviada, respuesta pendiente', 'Falta confirmación de coordinación']], 'documentos_referencia.csv': [['fuente_id', 'titulo', 'tipo', 'contenido'], ['DOC-SIM-01', 'Guía operativa simulada de consulta de antecedentes', 'documento_simulado', 'Para elaborar un resumen, recuperar antecedentes desde las fuentes autorizadas y conservar el identificador de la fuente. Los registros sin datos deben presentarse como información no disponible, no como actividades pendientes.'], ['DOC-SIM-02', 'Procedimiento simulado de borradores', 'documento_simulado', 'Los borradores de gestión se guardan por solicitud explícita del usuario autorizado. La generación de un borrador no modifica los registros originales ni representa una decisión clínica.'], ['DOC-SIM-03', 'Protocolo simulado de revisión administrativa', 'documento_simulado', 'La revisión administrativa se fundamenta solamente en reglas declaradas para la demostración. Se informa al profesional para validación. El sistema no asigna prioridades clínicas ni recomienda tratamientos.']]}
CRITERIOS_INICIALES = {'tipo_datos': 'COMPLETAMENTE_FICTICIOS', 'version': '1.0', 'alcance': 'Demostración de apoyo a revisión administrativa; sin efectos clínicos', 'reglas': [{'id': 'ADM-01', 'campo': 'estado_interconsulta', 'operador': 'igual', 'valor': 'pendiente', 'descripcion': 'Interconsulta registrada como pendiente'}, {'id': 'ADM-02', 'campo': 'estado_traslado', 'operador': 'en', 'valores': ['pendiente', 'no_realizado'], 'descripcion': 'Traslado con pendiente o no realizado expresamente registrado'}, {'id': 'ADM-03', 'campo': 'gestion_1_estado', 'operador': 'igual', 'valor': 'pendiente', 'descripcion': 'Gestión administrativa de seguimiento registrada como pendiente'}, {'id': 'ADM-04', 'campo': 'gestion_2_estado', 'operador': 'igual', 'valor': 'pendiente', 'descripcion': 'Segunda gestión administrativa registrada como pendiente'}], 'advertencia': 'Reglas SIMULADAS y no aprobadas por el INGER. No equivalen a priorización clínica.'}
MANIFIESTO_INICIAL = {'nombre': 'Base de demostración INGER', 'tipo_datos': 'COMPLETAMENTE_FICTICIOS', 'version': '1.0', 'archivos': ['ortogeriatria_2024.csv', 'ortogeriatria_2025.csv', 'interconsultas_traslados.csv', 'documentos_referencia.csv', 'criterios_revision_demo.json']}

for nombre, contenido in FUENTES_INICIALES.items():
    ruta = DATA_DRIVE / nombre
    if not ruta.exists():
        with ruta.open('w', newline='', encoding='utf-8-sig') as archivo:
            escritor = csv.writer(archivo)
            escritor.writerows(contenido)

for nombre, objeto in [('criterios_revision_demo.json', CRITERIOS_INICIALES),
                      ('MANIFIESTO_DATOS_SIMULADOS.json', MANIFIESTO_INICIAL)]:
    ruta = DATA_DRIVE / nombre
    if not ruta.exists():
        ruta.write_text(json.dumps(objeto, ensure_ascii=False, indent=2), encoding='utf-8')

print('Fuentes disponibles:', ', '.join(sorted(p.name for p in DATA_DRIVE.iterdir())))

Fuentes disponibles: MANIFIESTO_DATOS_SIMULADOS.json, criterios_revision_demo.json, documentos_referencia.csv, interconsultas_traslados.csv, ortogeriatria_2024.csv, ortogeriatria_2025.csv


## 05 — Carga, control de integridad y construcción de conocimiento
Se usan únicamente columnas permitidas, códigos ficticios `CASO-NNN` y documentación marcada como simulada. Cada antecedente conserva su identificador de fuente para poder explicar de dónde proviene una respuesta.

In [6]:
import pandas as pd

ARCHIVOS_ESPERADOS = {
    'ortogeriatria_2024.csv': ['caso_id','fecha_alta','fecha_control','motivo_control','gestion_1_estado','gestion_1_detalle','gestion_2_estado','gestion_2_detalle'],
    'ortogeriatria_2025.csv': ['caso_id','fecha_alta','fecha_control','motivo_control','gestion_1_estado','gestion_1_detalle','gestion_2_estado','gestion_2_detalle'],
    'interconsultas_traslados.csv': ['caso_id','estado_interconsulta','estado_traslado','dias_espera_registrados','observacion_interconsulta','observacion_traslado'],
    'documentos_referencia.csv': ['fuente_id','titulo','tipo','contenido'],
}


def leer_csv_como_dict(nombre, campos):
    with (DATA_DRIVE/nombre).open('r', encoding='utf-8-sig', newline='') as archivo:
        reader = csv.DictReader(archivo)
        if reader.fieldnames != campos:
            raise ValueError(f'Columnas inválidas en {nombre}. Se esperan exactamente: {campos}')
        datos = [{k:(v or '').strip() for k,v in fila.items()} for fila in reader]
        for fila in datos:
            if nombre != 'documentos_referencia.csv' and not re.fullmatch(r'CASO-\d{3}', fila['caso_id']):
                raise ValueError(f'ID no ficticio en {nombre}: {fila["caso_id"]}')
        return datos

manifiesto = json.loads((DATA_DRIVE/'MANIFIESTO_DATOS_SIMULADOS.json').read_text(encoding='utf-8'))
if manifiesto.get('tipo_datos') != 'COMPLETAMENTE_FICTICIOS':
    raise ValueError('El notebook solo admite bases señaladas como completamente ficticias.')
reglas_demo = json.loads((DATA_DRIVE/'criterios_revision_demo.json').read_text(encoding='utf-8'))
if reglas_demo.get('tipo_datos') != 'COMPLETAMENTE_FICTICIOS':
    raise ValueError('Los criterios deben estar identificados como simulados.')

orto2024 = leer_csv_como_dict('ortogeriatria_2024.csv', ARCHIVOS_ESPERADOS['ortogeriatria_2024.csv'])
orto2025 = leer_csv_como_dict('ortogeriatria_2025.csv', ARCHIVOS_ESPERADOS['ortogeriatria_2025.csv'])
interconsultas = leer_csv_como_dict('interconsultas_traslados.csv', ARCHIVOS_ESPERADOS['interconsultas_traslados.csv'])
documentos = leer_csv_como_dict('documentos_referencia.csv', ARCHIVOS_ESPERADOS['documentos_referencia.csv'])

# Guardarraíl adicional: no se aceptan identificadores personales en los campos textuales de la demo.
texto_corpus = json.dumps([orto2024,orto2025,interconsultas,documentos], ensure_ascii=False)
if re.search(r'\b\d{1,2}\.\d{3}\.\d{3}[-–][\dkK]\b', texto_corpus):
    raise ValueError('Se detectó un posible RUT; use exclusivamente datos simulados.')

casos = {}
for anio, registros in ((2024,orto2024),(2025,orto2025)):
    for fila in registros:
        cid=fila['caso_id']
        if cid in casos:
            raise ValueError('Identificador duplicado de seguimiento: '+cid)
        casos[cid]={'anio':anio, 'seguimiento':fila, 'interconsulta':None}
for fila in interconsultas:
    cid=fila['caso_id']
    if cid not in casos:
        raise ValueError('Registro interconsulta sin caso de seguimiento: '+cid)
    casos[cid]['interconsulta']=fila

fragmentos=[]
for cid, caso in casos.items():
    anio=caso['anio']
    seg=caso['seguimiento']
    fragmentos.append({
       'fuente_id':f'ORT-{anio}-{cid}', 'caso_id':cid, 'categoria':'seguimiento', 'titulo':f'Seguimiento {anio}',
       'texto':f"{cid}. Alta {seg['fecha_alta']}. Control {seg['fecha_control']}. Motivo: {seg['motivo_control']}. "
               f"Gestión 1: {seg['gestion_1_estado']} — {seg['gestion_1_detalle']}. "
               f"Gestión 2: {seg['gestion_2_estado'] or 'sin_registro'} — {seg['gestion_2_detalle'] or 'sin dato'}."
    })
    inter=caso['interconsulta']
    if inter:
        fragmentos.append({
           'fuente_id':f'INT-{cid}', 'caso_id':cid, 'categoria':'interconsultas_traslados',
           'titulo':'Interconsulta y traslado',
           'texto':f"{cid}. Interconsulta={inter['estado_interconsulta']}. Traslado={inter['estado_traslado']}. "
                   f"Días de espera registrados={inter['dias_espera_registrados'] or 'sin dato'}. "
                   f"Observación interconsulta={inter['observacion_interconsulta']}. "
                   f"Observación traslado={inter['observacion_traslado']}."
        })
for doc in documentos:
    if doc['tipo'] != 'documento_simulado':
        raise ValueError('Solo se admiten documentos de referencia simulados en este entorno.')
    fragmentos.append({'fuente_id':doc['fuente_id'],'caso_id':None,'categoria':'documento_simulado',
                       'titulo':doc['titulo'],'texto':doc['contenido']})

print(f'Casos ficticios: {len(casos)} | Fragmentos indexables: {len(fragmentos)} | Documentos simulados: {len(documentos)}')
display(pd.DataFrame([{'Caso':c,'Año':v['anio'],
                        'Interconsulta':(v['interconsulta'] or {}).get('estado_interconsulta','sin_registro'),
                        'Traslado':(v['interconsulta'] or {}).get('estado_traslado','sin_registro')}
                       for c,v in sorted(casos.items())]))

Casos ficticios: 8 | Fragmentos indexables: 19 | Documentos simulados: 3


,Caso,Año,Interconsulta,Traslado
0,CASO-001,2024,resuelta,realizado
1,CASO-002,2025,pendiente,pendiente
2,CASO-003,2024,sin_registro,sin_registro
3,CASO-004,2025,en_evaluacion,pendiente
4,CASO-005,2024,resuelta,realizado
5,CASO-006,2025,resuelta,realizado
6,CASO-007,2025,resuelta,no_aplica
7,CASO-008,2024,pendiente,no_realizado


## 06 — Recuperación semántica FAISS con umbral
Los registros se recuperan exactamente por ID `CASO-NNN` y los documentos mediante similitud semántica, filtrando por categoría. Se añade un **umbral de pertinencia configurable** para evitar que una búsqueda general sin coincidencias produzca respuestas sobre fuentes irrelevantes. El índice está cacheado en Google Drive.

In [7]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBEDDINGS_ID = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
UMBRAL_SEMANTICO = 0.25  # Umbral conservador para la demostración; ajustar tras medir Precision@k.
modelo_embeddings = SentenceTransformer(EMBEDDINGS_ID)

hash_fuentes = hashlib.sha256(json.dumps({'modelo':EMBEDDINGS_ID,'fragmentos':fragmentos}, sort_keys=True,
                                         ensure_ascii=False).encode('utf-8')).hexdigest()
indice_drive = CACHE_DRIVE / 'indice.faiss'
manifest_drive = CACHE_DRIVE / 'indice_manifest.json'
reutilizar = False
if indice_drive.exists() and manifest_drive.exists():
    try:
        anterior = json.loads(manifest_drive.read_text(encoding='utf-8'))
        reutilizar = anterior.get('huella') == hash_fuentes
    except (ValueError, OSError):
        reutilizar = False

if reutilizar:
    indice = faiss.read_index(str(indice_drive))
    print('Índice semántico restaurado desde Drive.')
else:
    textos = [f['titulo']+' '+f['texto'] for f in fragmentos]
    vectores = modelo_embeddings.encode(textos, normalize_embeddings=True,
                                       convert_to_numpy=True, show_progress_bar=False).astype('float32')
    indice = faiss.IndexFlatIP(vectores.shape[1])
    indice.add(vectores)
    temporal = LOCAL / 'indice.faiss'
    faiss.write_index(indice, str(temporal))
    shutil.copy2(temporal, indice_drive)
    manifest_drive.write_text(json.dumps({'huella':hash_fuentes,'modelo':EMBEDDINGS_ID,
                                          'total':len(fragmentos),'version':'2.0'},indent=2),encoding='utf-8')
    print('Nuevo índice semántico almacenado en Drive.')
assert indice.ntotal == len(fragmentos)

def buscar_semantico(consulta, caso_id=None, categoria=None, k=5, umbral=UMBRAL_SEMANTICO):
    permitidos = {i for i,f in enumerate(fragmentos)
                  if (not caso_id or f['caso_id']==caso_id) and (not categoria or f['categoria']==categoria)}
    if not permitidos:
        return []
    # La búsqueda por identificador es exacta; no se penaliza con un umbral semántico.
    if caso_id:
        return [dict(fragmentos[i], similitud=1.0, modo_recuperacion='identificador')
                for i in sorted(permitidos)]
    consulta_vector = modelo_embeddings.encode([consulta],normalize_embeddings=True,
                                                convert_to_numpy=True).astype('float32')
    scores, ids = indice.search(consulta_vector, len(fragmentos))
    return [dict(fragmentos[int(i)], similitud=round(float(puntaje),4),
                 modo_recuperacion='semantico')
            for puntaje,i in zip(scores[0],ids[0])
            if int(i) in permitidos and float(puntaje)>=umbral][:k]

print('FAISS:', indice.ntotal, 'fragmentos | Umbral semántico:', UMBRAL_SEMANTICO)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Índice semántico restaurado desde Drive.
FAISS: 19 fragmentos | Umbral semántico: 0.25


## 07 — Persistencia y memoria sin conflictos SQLite
La memoria administrativa y el **checkpointer de LangGraph** utilizan **bases SQLite y conexiones diferentes**. Ambos archivos se guardan primero en disco local y después se sincronizan por separado con Drive mediante la API `backup()` de SQLite, evitando copiar bases abiertas sin una instantánea consistente.

Cuando existe un archivo antiguo `inger_memoria.sqlite`, se conserva una copia de migración en las nuevas rutas. No se requiere intervención manual. La memoria contextual recupera el último caso de un hilo y utiliza embeddings para buscar recuerdos previos, sin exponer datos personales.

In [8]:
RUTA_LEGACY = STATE_DRIVE / 'inger_memoria.sqlite'
RUTA_APP_DRIVE = STATE_DRIVE / 'inger_administrativo.sqlite'
RUTA_GRAFO_DRIVE = STATE_DRIVE / 'inger_grafo.sqlite'
RUTA_APP_LOCAL = LOCAL / 'inger_administrativo.sqlite'
RUTA_GRAFO_LOCAL = LOCAL / 'inger_grafo.sqlite'

# Migración conservadora de la versión anterior: se copian las tablas históricas a
# bases independientes, evitando el uso compartido de una conexión entre subsistemas.
for destino, local in [(RUTA_APP_DRIVE, RUTA_APP_LOCAL), (RUTA_GRAFO_DRIVE, RUTA_GRAFO_LOCAL)]:
    origen = destino if destino.exists() else RUTA_LEGACY if RUTA_LEGACY.exists() else None
    if origen and not local.exists():
        origen_con = sqlite3.connect(str(origen))
        temporal_con = sqlite3.connect(str(local))
        try:
            origen_con.backup(temporal_con)
        finally:
            temporal_con.close()
            origen_con.close()

conexion_agente = sqlite3.connect(str(RUTA_APP_LOCAL), check_same_thread=False, timeout=30)
conexion_agente.execute('PRAGMA busy_timeout=30000')
conexion_agente.execute("""CREATE TABLE IF NOT EXISTS memorias (
    id INTEGER PRIMARY KEY AUTOINCREMENT, hilo TEXT NOT NULL, caso_id TEXT,
    intencion TEXT NOT NULL, resumen TEXT NOT NULL, creado_utc TEXT NOT NULL)""")
conexion_agente.execute("""CREATE TABLE IF NOT EXISTS borradores (
    id INTEGER PRIMARY KEY AUTOINCREMENT, idempotencia TEXT UNIQUE, hilo TEXT NOT NULL,
    caso_id TEXT NOT NULL, contenido TEXT NOT NULL, creado_utc TEXT NOT NULL)""")
conexion_agente.commit()

# Se crea una conexión distinta para LangGraph en la celda de orquestación.
conexion_grafo = None

def backup_sqlite(conexion, destino_drive, etiqueta):
    temporal = LOCAL / (etiqueta + '_snapshot.sqlite')
    if temporal.exists():
        temporal.unlink()
    destino_local = sqlite3.connect(str(temporal))
    try:
        conexion.backup(destino_local)
    finally:
        destino_local.close()
    destino_preparado = destino_drive.with_suffix('.sqlite.tmp')
    shutil.copy2(temporal, destino_preparado)
    destino_preparado.replace(destino_drive)

def guardar_snapshot():
    backup_sqlite(conexion_agente, RUTA_APP_DRIVE, 'administrativo')
    if conexion_grafo is not None:
        backup_sqlite(conexion_grafo, RUTA_GRAFO_DRIVE, 'grafo')

def registrar_memoria(hilo, caso_id, intencion, resumen):
    with conexion_agente:
        conexion_agente.execute(
            'INSERT INTO memorias(hilo,caso_id,intencion,resumen,creado_utc) VALUES(?,?,?,?,?)',
            (hilo,caso_id,intencion,resumen[:400],datetime.now(timezone.utc).isoformat()))

def ultimo_caso(hilo):
    fila = conexion_agente.execute(
        'SELECT caso_id FROM memorias WHERE hilo=? AND caso_id IS NOT NULL ORDER BY id DESC LIMIT 1',
        (hilo,)).fetchone()
    return fila[0] if fila else None

def recuperar_memoria_semantica(hilo, pregunta, k=3):
    filas = conexion_agente.execute(
        'SELECT caso_id,intencion,resumen,creado_utc FROM memorias WHERE hilo=? ORDER BY id DESC LIMIT 30',
        (hilo,)).fetchall()
    if not filas:
        return []
    vectores = modelo_embeddings.encode([x[2] for x in filas],normalize_embeddings=True,
                                        convert_to_numpy=True).astype('float32')
    consulta = modelo_embeddings.encode([pregunta],normalize_embeddings=True,
                                        convert_to_numpy=True).astype('float32')[0]
    puntajes = vectores @ consulta
    elegidos = sorted(range(len(filas)), key=lambda i:float(puntajes[i]), reverse=True)[:k]
    return [{'caso_id':filas[i][0],'intencion':filas[i][1],'resumen':filas[i][2],
             'creado_utc':filas[i][3]} for i in elegidos]

def guardar_borrador_sql(hilo, caso_id, texto, solicitud):
    clave = hashlib.sha256((hilo+'|'+caso_id+'|'+solicitud).encode('utf-8')).hexdigest()
    with conexion_agente:
        conexion_agente.execute(
            'INSERT OR IGNORE INTO borradores(idempotencia,hilo,caso_id,contenido,creado_utc) VALUES(?,?,?,?,?)',
            (clave,hilo,caso_id,texto[:4500],datetime.now(timezone.utc).isoformat()))
    fila = conexion_agente.execute('SELECT id FROM borradores WHERE idempotencia=?',(clave,)).fetchone()
    return {'permitido':True,'borrador_id':fila[0],'caso_id':caso_id,'idempotente':True}

print('Memorias separadas: SQLite administrativo + SQLite LangGraph (sin conexión compartida).')

Memorias separadas: SQLite administrativo + SQLite LangGraph (sin conexión compartida).


## 08 — Reglas y herramientas de operación
El agente dispone de consulta exacta, búsqueda semántica, revisión administrativa simulada y escritura de borradores. La escritura exige el rol `gestor_demo` y una solicitud explícita; no modifica fuentes de origen.

In [9]:
from langchain_core.tools import tool

PATRON_CASO = re.compile(r'\bCASO[\s-]?(\d{3})\b',re.I)

def extraer_caso(pregunta):
    m=PATRON_CASO.search(pregunta or '')
    return 'CASO-'+m.group(1) if m else None

def es_referencia_conversacional(pregunta):
    t=pregunta.casefold()
    return any(x in t for x in ['sus pendientes','sus gestiones','ese caso','este caso','el mismo caso','¿y su','y su traslado','¿y los pendientes'])

def detectar_intencion(pregunta):
    t=pregunta.casefold()
    if any(x in t for x in ['guardar borrador','guarda borrador','registrar borrador','registra borrador']):
        return 'borrador'
    if any(x in t for x in ['protocolo','documento','normativa','procedimiento','guía operativa']):
        return 'documentos'
    if any(x in t for x in ['prioridad','priorizar','prioriz','qué casos revisar','revision administrativa','revisión administrativa']):
        return 'revision'
    if any(x in t for x in ['pendiente','sin resolver','inconsistencia','espera','no realizado']):
        return 'pendientes'
    return 'resumen'

def aplicar_reglas(caso_id):
    caso=casos.get(caso_id)
    if not caso: return {'caso_id':caso_id,'estado':'no_encontrado','reglas_activadas':[]}
    seguimiento=caso['seguimiento']; inter=caso['interconsulta'] or {}
    hechos={**seguimiento, **inter}
    activadas=[]
    for regla in reglas_demo['reglas']:
        valor=hechos.get(regla['campo'],'')
        if regla['operador']=='igual': coincide=(valor==regla['valor'])
        elif regla['operador']=='en': coincide=(valor in regla['valores'])
        else: raise ValueError('Operador de regla no autorizado')
        if coincide: activadas.append({'id':regla['id'],'descripcion':regla['descripcion'],'evidencia':regla['campo']+'='+valor,
                            'fuente_id':('INT-'+caso_id if regla['campo'] in ('estado_interconsulta','estado_traslado')
                                         else 'ORT-'+str(caso['anio'])+'-'+caso_id)})
    return {'caso_id':caso_id,'revision_administrativa':bool(activadas),'reglas_activadas':activadas,
            'advertencia':reglas_demo['advertencia']}

@tool
def consultar_registros(caso_id: str) -> str:
    """Recupera antecedentes ficticios por identificador CASO-NNN; no consulta datos personales."""
    return json.dumps([f for f in fragmentos if f['caso_id']==caso_id], ensure_ascii=False)

@tool
def buscar_conocimiento(consulta: str, categoria: str = '', caso_id: str = '') -> str:
    """Busca fragmentos por similitud semántica con filtros de categoría o caso."""
    resultado=buscar_semantico(consulta,caso_id or None,categoria or None)
    return json.dumps(resultado,ensure_ascii=False)

@tool
def revisar_caso(caso_id: str) -> str:
    """Aplica exclusivamente reglas administrativas SIMULADAS y devuelve las evidencias de activación."""
    return json.dumps(aplicar_reglas(caso_id),ensure_ascii=False)

@tool
def registrar_borrador(hilo: str, caso_id: str, texto: str, solicitud: str, rol: str) -> str:
    """Guarda un borrador local solo con rol autorizado y petición de escritura explícita."""
    if rol!='gestor_demo': return json.dumps({'permitido':False,'motivo':'Rol sin permiso de escritura.'})
    if detectar_intencion(solicitud)!='borrador': return json.dumps({'permitido':False,'motivo':'Falta orden expresa de guardar borrador.'})
    if caso_id not in casos: return json.dumps({'permitido':False,'motivo':'Caso no encontrado.'})
    return json.dumps(guardar_borrador_sql(hilo,caso_id,texto,solicitud),ensure_ascii=False)

@tool
def buscar_documentos_lexicos(consulta: str) -> str:
    """Recuperación alternativa con coincidencia de términos explícitos en documentos simulados."""
    import unicodedata
    def terminos(texto):
        normal=unicodedata.normalize('NFKD', texto.casefold())
        normal=''.join(c for c in normal if not unicodedata.combining(c))
        palabras=re.findall(r'\w+',normal)
        return {p[:-1] if p.endswith('s') and len(p)>=5 else p for p in palabras}
    excluidas={'sobre','cuando','para','desde','donde','cuale','dice','establece',
               'documento','protocolo','simulado','este','esta','referencia'}
    claves={p for p in terminos(consulta) if len(p)>=4 and p not in excluidas}
    coincidencias=[]
    for f in fragmentos:
        if f['categoria']!='documento_simulado': continue
        interseccion=claves & terminos(f['titulo']+' '+f['texto'])
        if interseccion:
            coincidencias.append((len(interseccion),dict(f,coincidencias_lexicas=sorted(interseccion),
                                                    modo_recuperacion='lexico')))
    if not coincidencias: return '[]'
    mejor=max(n for n,_ in coincidencias)
    return json.dumps([f for n,f in coincidencias if n==mejor][:5],ensure_ascii=False)

HERRAMIENTAS = {t.name:t for t in (consultar_registros,buscar_conocimiento,revisar_caso,registrar_borrador,buscar_documentos_lexicos)}
print('Herramientas disponibles:', ', '.join(HERRAMIENTAS))

Herramientas disponibles: consultar_registros, buscar_conocimiento, revisar_caso, registrar_borrador, buscar_documentos_lexicos


## 09 — Agente LangGraph: planificación ejecutable y evidencia verificable
Groq devuelve un plan estructurado (`intención`, `herramientas` y `objetivo`). El sistema verifica el plan contra políticas de seguridad, crea tareas reales y las ejecuta **una por una**, almacenando `pendientes` y `ejecutadas` en el estado de LangGraph. Una recuperación insuficiente puede añadir una tarea de recuperación complementaria; si no hay fuentes, el agente se abstiene.

Antes de guardar cualquier borrador, un verificador independiente construye la respuesta a partir de campos y fragmentos **originales**. Groq selecciona fuentes, pero no puede insertar libremente fechas, estados, procedimientos o recomendaciones en el texto final. Si Groq falla en la planificación, un plan conservador de respaldo queda identificado como tal en el registro (y no se cuenta como autonomía demostrada). La escritura se hace **después** de validar la respuesta.

In [10]:
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from uuid import uuid4
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_groq import ChatGroq

class EstadoINGER(TypedDict, total=False):
    messages: Annotated[list[BaseMessage], add_messages]
    pregunta: str
    hilo: str
    rol: str
    caso_id: str | None
    intencion: str
    origen_plan: str
    objetivo: str
    plan: list[dict]
    pendientes: list[dict]
    ejecutadas: list[dict]
    evidencia: list[dict]
    reglas: list[dict]
    seleccion: list[str]
    validacion: dict
    respuesta: str
    escritura: dict | None
    ruta: str
    revisiones: int
    alertas: list[str]

class PlanGroq(BaseModel):
    intencion: Literal['resumen','pendientes','revision','documentos','borrador']
    herramientas: list[Literal['consultar_registros','buscar_conocimiento','revisar_caso']]
    objetivo: str

class SeleccionGroq(BaseModel):
    fuentes: list[str]
    foco: Literal['antecedentes','pendientes','revision','documentos','borrador','sin_evidencia']

llm = ChatGroq(model=os.environ['GROQ_MODEL'],temperature=0,max_retries=3,timeout=90)
planificador_llm = llm.with_structured_output(PlanGroq, method='json_schema', strict=True)
selector_llm = llm.with_structured_output(SeleccionGroq, method='json_schema', strict=True)

REGLAS_PLAN = """Eres el planificador de un asistente institucional que trabaja SOLO con datos ficticios.
Selecciona una intención y herramientas de SOLO lectura, en el orden lógico para resolver la consulta.
Herramientas: consultar_registros (caso exacto), buscar_conocimiento (semántica/documentos),
revisar_caso (aplicar reglas administrativas simuladas). NO dispones de una herramienta de escritura
porque la escritura se autoriza y ejecuta en un nodo posterior de seguridad. No inventes casos.
Si se solicitan reglas/criterios administrativos, selecciona revisar_caso; si se pide documentación,
selecciona buscar_conocimiento; si se solicita un caso exacto, selecciona consultar_registros.
Devuelve un plan breve, secuenciado, con las herramientas estrictamente necesarias."""

# Conexión exclusiva del checkpointer, sin transacciones compartidas con los borradores.
conexion_grafo = sqlite3.connect(str(RUTA_GRAFO_LOCAL), check_same_thread=False, timeout=30)
conexion_grafo.execute('PRAGMA busy_timeout=30000')
checkpointer = SqliteSaver(conexion_grafo)


def construir_plan_seguro(intencion, herramientas, caso_id, pregunta):
    tareas=[]
    permitidas={'consultar_registros','buscar_conocimiento','revisar_caso'}
    seleccionadas=[t for t in herramientas if t in permitidas]
    if intencion=='documentos':
        seleccionadas=['buscar_conocimiento']
    elif intencion=='revision':
        seleccionadas=['revisar_caso']
    elif caso_id:
        seleccionadas=['consultar_registros']+[t for t in seleccionadas if t=='buscar_conocimiento']
    elif 'buscar_conocimiento' not in seleccionadas:
        seleccionadas=['buscar_conocimiento']
    for herramienta in dict.fromkeys(seleccionadas):
        if herramienta=='consultar_registros':
            if caso_id:
                tareas.append({'herramienta':herramienta,'caso_id':caso_id})
        elif herramienta=='revisar_caso':
            for cid in ([caso_id] if caso_id else sorted(casos)):
                tareas.append({'herramienta':herramienta,'caso_id':cid})
        elif herramienta=='buscar_conocimiento':
            tareas.append({'herramienta':herramienta,'consulta':pregunta,
                           'categoria':'documento_simulado' if intencion=='documentos' else '',
                           'caso_id':caso_id or ''})
    if len(tareas)>12:
        raise ValueError('El plan excede el límite de tareas autorizadas.')
    return tareas


def nodo_planificar(s: EstadoINGER):
    pregunta=s['pregunta']
    caso_id=extraer_caso(pregunta)
    if caso_id is None and es_referencia_conversacional(pregunta):
        caso_id=ultimo_caso(s['hilo'])
    intencion_detectada=detectar_intencion(pregunta)
    alertas=[]
    try:
        propuesta=planificador_llm.invoke([SystemMessage(content=REGLAS_PLAN),
             HumanMessage(content=f'Consulta: {pregunta}\nCaso contextual: {caso_id or "ninguno"}')])
        intencion=propuesta.intencion
        # El modelo decide las herramientas, pero la política impide cambiar una operación sensible.
        if intencion!=intencion_detectada:
            alertas.append('Intención armonizada con la regla operativa de seguridad.')
            intencion=intencion_detectada
        herramientas=propuesta.herramientas
        objetivo=propuesta.objetivo[:250]
        origen='groq'
    except Exception as exc:
        # La ejecución continúa con seguridad, pero el informe NO cuenta este caso como autonomía de Groq.
        intencion=intencion_detectada
        herramientas=['revisar_caso' if intencion=='revision' else
                      'buscar_conocimiento' if intencion=='documentos' or not caso_id else 'consultar_registros']
        objetivo='Consulta segura con reglas de respaldo'
        origen='respaldo_determinista'
        alertas.append(f'Planificador Groq no disponible: {type(exc).__name__}')
    plan=construir_plan_seguro(intencion,herramientas,caso_id,pregunta)
    return {'caso_id':caso_id,'intencion':intencion,'objetivo':objetivo,'origen_plan':origen,
            'plan':plan,'pendientes':plan,'ejecutadas':[],'evidencia':[],'reglas':[],
            'respuesta':'','seleccion':[],'validacion':{},'escritura':None,
            'ruta':'planificar','revisiones':0,'alertas':alertas}


def nodo_ejecutar_tarea(s: EstadoINGER):
    pendientes=list(s.get('pendientes',[]))
    if not pendientes:
        raise RuntimeError('No hay tareas que ejecutar.')
    tarea=pendientes.pop(0)
    nombre=tarea['herramienta']
    evidencias=list(s.get('evidencia',[]))
    reglas=list(s.get('reglas',[]))
    if nombre=='consultar_registros':
        nuevos=json.loads(consultar_registros.invoke({'caso_id':tarea['caso_id']}))
        evidencias+=nuevos
    elif nombre=='buscar_conocimiento':
        nuevos=json.loads(buscar_conocimiento.invoke({'consulta':tarea['consulta'],
                            'categoria':tarea['categoria'],'caso_id':tarea['caso_id']}))
        evidencias+=nuevos
    elif nombre=='revisar_caso':
        reglas.append(json.loads(revisar_caso.invoke({'caso_id':tarea['caso_id']})))
    elif nombre=='buscar_documentos_lexicos':
        evidencias+=json.loads(buscar_documentos_lexicos.invoke({'consulta':tarea['consulta']}))
    else:
        raise PermissionError('Herramienta no autorizada: '+str(nombre))
    evidencias=list({f['fuente_id']:f for f in evidencias}.values())
    ejecutadas=list(s.get('ejecutadas',[]))+[tarea]
    return {'pendientes':pendientes,'ejecutadas':ejecutadas,
            'evidencia':evidencias,'reglas':reglas,'ruta':'ejecutar:'+nombre}


def nodo_evaluar_evidencia(s: EstadoINGER):
    evidencia=s.get('evidencia',[])
    reglas=s.get('reglas',[])
    # No se amplía la búsqueda exacta de un expediente inexistente a otros pacientes.
    if evidencia or reglas or s.get('caso_id'):
        return {'ruta':'evidencia_suficiente' if (evidencia or reglas) else 'sin_evidencia'}
    # Ante una consulta documental sin similitud suficiente, cambiar a coincidencia léxica.
    if s.get('intencion')=='documentos' and s.get('revisiones',0)==0:
        nueva={'herramienta':'buscar_documentos_lexicos','consulta':s['pregunta']}
        return {'pendientes':[nueva],'plan':list(s['plan'])+[nueva],
                'revisiones':1,'ruta':'replanificacion_lexica'}
    return {'ruta':'sin_evidencia'}


def ruta_evaluacion(s: EstadoINGER):
    if s.get('pendientes'): return 'continuar'
    return 'generar' if (s.get('evidencia') or s.get('reglas')) else 'sin_evidencia'


def nodo_sin_evidencia(s: EstadoINGER):
    return {'respuesta':'No existen antecedentes suficientes en las fuentes ficticias consultadas.',
            'seleccion':[],'ruta':'sin_evidencia','validacion':{'aprobada':False,'motivo':'sin evidencia'}}


def nodo_seleccionar_con_groq(s: EstadoINGER):
    fuentes={f['fuente_id'] for f in s.get('evidencia',[])}
    fuentes.update(regla['fuente_id'] for item in s.get('reglas',[])
                   for regla in item.get('reglas_activadas',[]))
    fuentes=sorted(fuentes)
    if not fuentes:
        return {'seleccion':[],'ruta':'sin_fuentes'}
    try:
        seleccion=selector_llm.invoke([
            SystemMessage(content='Selecciona SOLO identificadores de fuente presentes en la lista autorizada. '
                                  'No redactes hechos, fechas ni recomendaciones; devuelve fuentes y foco.'),
            HumanMessage(content=json.dumps({'consulta':s['pregunta'],'intencion':s['intencion'],
                    'fuentes_disponibles':fuentes,
                    'fragmentos':s.get('evidencia',[])[:8],
                    'criterios':s.get('reglas',[])[:8]},ensure_ascii=False))])
        seleccion_validada=[f for f in seleccion.fuentes if f in fuentes]
        if s.get('caso_id') and s['intencion'] in ('resumen','pendientes','borrador'):
            # Una síntesis de caso debe mantener todos sus antecedentes exactos.
            seleccion_validada=fuentes
        if not seleccion_validada:
            seleccion_validada=fuentes
        return {'seleccion':seleccion_validada,'ruta':'seleccion_groq'}
    except Exception as exc:
        return {'seleccion':fuentes,'ruta':'seleccion_respaldo',
                'alertas':list(s.get('alertas',[]))+[f'Selección Groq falló: {type(exc).__name__}']}


def representar_fuentes_verificadas(intencion, evidencia, reglas, caso_id, seleccion):
    # Importante: las oraciones fácticas se generan exclusivamente desde los registros originales.
    # No hay superficie libre en la que Groq pueda introducir fechas o estados inventados.
    ids_validos={f['fuente_id'] for f in evidencia}
    ids_validos.update(r['fuente_id'] for item in reglas for r in item.get('reglas_activadas',[]))
    autorizadas=set(seleccion) & ids_validos
    fragmentos_permitidos=[f for f in evidencia if f['fuente_id'] in autorizadas]
    lineas=[]
    if intencion=='borrador':
        lineas.append('BORRADOR ADMINISTRATIVO — sujeto a revisión profesional.')
    elif intencion=='revision':
        lineas.append('REVISIÓN ADMINISTRATIVA — criterios de demostración, no clínicos.')
    elif intencion=='documentos':
        lineas.append('CONSULTA DOCUMENTAL — referencias exclusivamente simuladas.')
    else:
        lineas.append('ANTECEDENTES DOCUMENTADOS — fuentes de demostración.')
    if caso_id:
        lineas.append('Caso: '+caso_id)
    if intencion=='revision':
        for item in reglas:
            activas=[r for r in item.get('reglas_activadas',[]) if r['fuente_id'] in autorizadas]
            if activas:
                lineas.append(f"Caso {item['caso_id']}: revisión administrativa indicada por reglas simuladas.")
                for regla in activas:
                    lineas.append(f"- {regla['id']}: {regla['descripcion']} | Dato: {regla['evidencia']} | Fuente: {regla['fuente_id']}")
            elif not item.get('reglas_activadas'):
                lineas.append(f"Caso {item['caso_id']}: sin reglas administrativas activadas.")
    else:
        for f in fragmentos_permitidos:
            if intencion=='pendientes' and f.get('caso_id') in casos:
                cid=f['caso_id']; caso=casos[cid]; seg=caso['seguimiento']; inter=caso['interconsulta'] or {}
                if f['categoria']=='seguimiento':
                    for n in (1,2):
                        estado=seg.get(f'gestion_{n}_estado','')
                        if estado=='pendiente':
                            lineas.append(f"Gestión {n}: pendiente registrado | {seg[f'gestion_{n}_detalle']} | Fuente: {f['fuente_id']}")
                        elif estado in ('sin_registro',''):
                            lineas.append(f"Gestión {n}: sin registro; no se presume pendiente | Fuente: {f['fuente_id']}")
                if f['categoria']=='interconsultas_traslados':
                    for nombre,campo in [('Interconsulta','estado_interconsulta'),('Traslado','estado_traslado')]:
                        estado=inter.get(campo,'sin_registro')
                        if estado in ('pendiente','no_realizado'):
                            lineas.append(f'{nombre}: {estado} expresamente registrado | Fuente: {f["fuente_id"]}')
                        elif estado=='sin_registro':
                            lineas.append(f'{nombre}: sin registro; no se presume pendiente | Fuente: {f["fuente_id"]}')
            else:
                lineas.append(f"[{f['fuente_id']}] {f['titulo']}: {f['texto']}")
    if intencion=='pendientes' and len(lineas)<4:
        lineas.append('No se identificaron pendientes explícitos adicionales en los registros consultados.')
    lineas.append('Fuentes verificadas: '+(', '.join(sorted(autorizadas)) if autorizadas else 'ninguna'))
    lineas.append('Control: datos ficticios; confirmación profesional requerida; sin decisiones clínicas.')
    return '\n'.join(lineas)


def nodo_verificar(s: EstadoINGER):
    # El generador no confía en texto libre del LLM: solo acepta IDs que existen en la evidencia.
    disponibles={f['fuente_id'] for f in s.get('evidencia',[])}
    disponibles.update(r['fuente_id'] for item in s.get('reglas',[])
                       for r in item.get('reglas_activadas',[]))
    seleccion=[x for x in s.get('seleccion',[]) if x in disponibles]
    if not seleccion and disponibles:
        seleccion=sorted(disponibles)
    if not s.get('evidencia') and not s.get('reglas'):
        return {'respuesta':'No existen antecedentes suficientes en las fuentes ficticias consultadas.',
                'validacion':{'aprobada':False,'fuentes':[],'modo':'abstencion'},'ruta':'sin_evidencia'}
    texto=representar_fuentes_verificadas(s['intencion'],s.get('evidencia',[]),s.get('reglas',[]),
                                         s.get('caso_id'),seleccion)
    return {'respuesta':texto,'seleccion':seleccion,
            'validacion':{'aprobada':True,'fuentes':seleccion,'modo':'render_factual_controlado'},
            'ruta':'respuesta_verificada'}


def nodo_guardar_borrador(s: EstadoINGER):
    if s['intencion']!='borrador':
        return {'escritura':None}
    if not s.get('validacion',{}).get('aprobada'):
        return {'escritura':{'permitido':False,'motivo':'Evidencia no validada.'},
                'respuesta':s['respuesta']+'\nBorrador no guardado: evidencia insuficiente.'}
    if not s.get('caso_id'):
        return {'escritura':{'permitido':False,'motivo':'Sin identificador del caso.'},
                'respuesta':s['respuesta']+'\nBorrador no guardado: caso no indicado.'}
    estado=json.loads(registrar_borrador.invoke({'hilo':s['hilo'],'caso_id':s['caso_id'],
                   'texto':s['respuesta'],'solicitud':s['pregunta'],'rol':s['rol']}))
    mensaje= ('\nBorrador administrativo guardado. ID='+str(estado['borrador_id'])
              if estado.get('permitido') else '\nBorrador NO guardado: '+estado.get('motivo','sin permiso'))
    return {'escritura':estado,'respuesta':s['respuesta']+mensaje,
            'ruta':'borrador_autorizado' if estado.get('permitido') else 'borrador_denegado'}


def nodo_memorizar(s: EstadoINGER):
    fuentes=','.join(s.get('seleccion',[])[:6])
    caso_memoria=s.get('caso_id') if s.get('caso_id') in casos else None
    registrar_memoria(s['hilo'],caso_memoria,s['intencion'],
        'caso='+str(s.get('caso_id'))+'; intencion='+s['intencion']+'; fuentes='+fuentes)
    return {'messages':[AIMessage(content=s['respuesta'])]}


grafo=StateGraph(EstadoINGER)
for nombre,func in [
    ('planificar',nodo_planificar),('ejecutar_tarea',nodo_ejecutar_tarea),
    ('evaluar_evidencia',nodo_evaluar_evidencia),('sin_evidencia',nodo_sin_evidencia),
    ('seleccionar_con_groq',nodo_seleccionar_con_groq),('verificar_y_renderizar',nodo_verificar),
    ('guardar_borrador',nodo_guardar_borrador),('memorizar',nodo_memorizar)]:
    grafo.add_node(nombre,func)
grafo.add_edge(START,'planificar')
grafo.add_conditional_edges('planificar',lambda s:'ejecutar' if s.get('pendientes') else 'evaluar',
                            {'ejecutar':'ejecutar_tarea','evaluar':'evaluar_evidencia'})
grafo.add_conditional_edges('ejecutar_tarea',lambda s:'continuar' if s.get('pendientes') else 'evaluar',
                            {'continuar':'ejecutar_tarea','evaluar':'evaluar_evidencia'})
grafo.add_conditional_edges('evaluar_evidencia',ruta_evaluacion,
                            {'continuar':'ejecutar_tarea','generar':'seleccionar_con_groq',
                             'sin_evidencia':'sin_evidencia'})
grafo.add_edge('seleccionar_con_groq','verificar_y_renderizar')
grafo.add_edge('sin_evidencia','verificar_y_renderizar')
grafo.add_conditional_edges('verificar_y_renderizar',
    lambda s:'guardar' if s['intencion']=='borrador' else 'memorizar',
    {'guardar':'guardar_borrador','memorizar':'memorizar'})
grafo.add_edge('guardar_borrador','memorizar')
grafo.add_edge('memorizar',END)
agente=grafo.compile(checkpointer=checkpointer)


def consultar_inger(pregunta,hilo='gerencia-demo',rol='lector_demo'):
    if not pregunta.strip():
        raise ValueError('La consulta no puede estar vacía.')
    entrada={'pregunta':pregunta,'hilo':hilo,'rol':rol,
             'messages':[HumanMessage(content=pregunta)]}
    config={'configurable':{'thread_id':hilo},'run_id':uuid4(),
            'run_name':'INGER_Consulta_Gestion','recursion_limit':36,
            'tags':['INGER','datos_ficticios','demostracion'],
            'metadata':{'unidad':'gestion_casos','tipo_datos':'ficticios'}}
    resultado=agente.invoke(entrada,config=config)
    guardar_snapshot()
    return resultado

print('Agente LangGraph compilado | Planner Groq estructurado | SQLite independiente | Control factual.')

Agente LangGraph compilado | Planner Groq estructurado | SQLite independiente | Control factual.


## 10 — Pruebas deterministas obligatorias
Se comprueban invariantes de datos, autorización, clasificación de pendientes, construcción de planes reales, control de fuentes, rechazo de fechas inventadas y aislamiento de los dos archivos SQLite. Estas pruebas no consumen tokens de Groq; si alguna falla, Colab detiene el proceso.

In [11]:
# Comprobaciones sin API (fallan explícitamente ante un defecto).
assert len(casos)==8 and len(fragmentos)==19
assert extraer_caso('Resumen del CASO-002')=='CASO-002'
assert extraer_caso('sin identificador') is None
assert detectar_intencion('Guardar borrador CASO-004')=='borrador'
assert detectar_intencion('Revisar el protocolo simulado')=='documentos'
assert detectar_intencion('¿Qué casos tienen prioridad?')=='revision'
assert aplicar_reglas('CASO-002')['revision_administrativa'] is True
assert aplicar_reglas('CASO-007')['revision_administrativa'] is False
assert casos['CASO-003']['interconsulta']['estado_interconsulta']=='sin_registro'
assert len(buscar_semantico('CASO-002',caso_id='CASO-002'))==2
assert all(x['categoria']=='documento_simulado' for x in buscar_semantico(
           'protocolo',categoria='documento_simulado',umbral=0.0))
assert json.loads(registrar_borrador.invoke({'hilo':'control','caso_id':'CASO-002',
    'texto':'texto ficticio','solicitud':'guardar borrador CASO-002','rol':'lector_demo'}))['permitido'] is False
assert all(x['herramienta']=='revisar_caso' for x in construir_plan_seguro('revision',
       ['buscar_conocimiento'],None,'¿Qué casos necesitan revisión?'))
assert len(construir_plan_seguro('revision',['revisar_caso'],None,'revisión'))==8
assert construir_plan_seguro('documentos',['consultar_registros'],None,'protocolo')[0]['herramienta']=='buscar_conocimiento'
assert len(construir_plan_seguro('resumen',['consultar_registros','buscar_conocimiento'],'CASO-002','Consulta CASO-002'))==2
assert RUTA_APP_LOCAL != RUTA_GRAFO_LOCAL
assert conexion_agente is not conexion_grafo
assert 'consultar_registros' in HERRAMIENTAS
assert len(json.loads(buscar_documentos_lexicos.invoke({'consulta':'protocolo simulado sobre registros'})))>=1
assert nodo_evaluar_evidencia({'intencion':'documentos','revisiones':0,'caso_id':None,'evidencia':[], 'reglas':[], 'pregunta':'protocolo simulado','plan':[]})['ruta']=='replanificacion_lexica'

# Inyección de fecha inventada al selector: no tiene superficie para aparecer en el texto final.
registro_caso=[f for f in fragmentos if f['caso_id']=='CASO-004']
texto_prueba=representar_fuentes_verificadas('borrador',registro_caso,[],'CASO-004',
                 [f['fuente_id'] for f in registro_caso]+['2025-04-25'])
assert '2025-04-25' not in texto_prueba
assert '2025-04-02' in texto_prueba and '2025-04-18' in texto_prueba

# Un estado sin registro no equivale a un pendiente.
registro_sin=[f for f in fragmentos if f['caso_id']=='CASO-003']
respuesta_sin=representar_fuentes_verificadas('pendientes',registro_sin,[],'CASO-003',
                    [f['fuente_id'] for f in registro_sin])
assert 'sin registro; no se presume pendiente' in respuesta_sin
assert 'pendiente registrado' not in respuesta_sin

print('25 controles locales aprobados: herramientas, planes, evidencias y seguridad.')

25 controles locales aprobados: herramientas, planes, evidencias y seguridad.


## 11 — Demostración automática de ocho escenarios
Se ejecutan ocho casos sin `input()`: síntesis, continuidad contextual, revisión de todos los casos, consulta documental, escritura permitida, escritura rechazada, caso inexistente y datos ausentes. Cada escenario tiene **criterios de aceptación específicos**; un error funcional no puede registrarse como éxito. Se guardan resultados aun si alguna prueba falla y, al final, se exige **8/8**.

In [12]:
ESCENARIOS = [
  {'nombre':'01 · Síntesis integral','pregunta':'Resume los antecedentes registrados del CASO-002 y cita sus fuentes.',
   'hilo':'gerencia-operativa-01','rol':'lector_demo','intencion':'resumen'},
  {'nombre':'02 · Memoria contextual','pregunta':'¿Y sus gestiones pendientes? Usa el mismo caso anterior.',
   'hilo':'gerencia-operativa-01','rol':'lector_demo','intencion':'pendientes'},
  {'nombre':'03 · Planificación adaptativa','pregunta':'¿Qué casos necesitan revisión administrativa según los criterios simulados?',
   'hilo':'gerencia-operativa-02','rol':'lector_demo','intencion':'revision'},
  {'nombre':'04 · Consulta documental','pregunta':'¿Qué establece el protocolo simulado sobre datos sin registro?',
   'hilo':'gerencia-operativa-03','rol':'lector_demo','intencion':'documentos'},
  {'nombre':'05 · Escritura autorizada','pregunta':'Guardar borrador de seguimiento administrativo del CASO-004 con antecedentes registrados.',
   'hilo':'gerencia-operativa-04','rol':'gestor_demo','intencion':'borrador'},
  {'nombre':'06 · Escritura denegada','pregunta':'Guardar borrador de seguimiento administrativo del CASO-002.',
   'hilo':'gerencia-operativa-05','rol':'lector_demo','intencion':'borrador'},
  {'nombre':'07 · Caso inexistente','pregunta':'Consulta los antecedentes del CASO-999.',
   'hilo':'gerencia-operativa-06','rol':'lector_demo','intencion':'resumen'},
  {'nombre':'08 · Ausencia de datos','pregunta':'Identifica pendientes del CASO-003 distinguiendo registros ausentes.',
   'hilo':'gerencia-operativa-07','rol':'lector_demo','intencion':'pendientes'},
]

INICIO_DEMO_UTC=datetime.now(timezone.utc)
RESULTADOS=[]
for indice_escena, escena in enumerate(ESCENARIOS,1):
    print('\n'+'═'*68+'\n'+escena['nombre']+'\n'+'═'*68)
    t0=time.perf_counter()
    try:
        res=consultar_inger(escena['pregunta'],hilo=escena['hilo'],rol=escena['rol'])
        texto=res['respuesta']
        assert isinstance(texto,str) and texto.strip(), 'Respuesta vacía.'
        assert res['intencion']==escena['intencion'], 'Clasificación de intención inesperada.'
        assert res['origen_plan']=='groq', 'No hay evidencia de planificación autónoma por Groq.'
        assert len(res.get('ejecutadas',[]))>=1, 'El plan no ejecutó herramientas.'
        assert not res.get('pendientes'), 'Quedaron tareas sin ejecutar.'
        assert 'Fuentes verificadas:' in texto or res.get('ruta')=='sin_evidencia', 'Faltan fuentes verificadas.'
        if indice_escena==1:
            assert 'INT-CASO-002' in texto and 'ORT-2025-CASO-002' in texto
        if indice_escena==2:
            assert res['caso_id']=='CASO-002' and 'pendiente' in texto.lower()
        if indice_escena==3:
            assert any(x['herramienta']=='revisar_caso' for x in res['ejecutadas'])
            assert len(res['reglas'])==8 and 'ADM-' in texto
        if indice_escena==4:
            assert any(x['categoria']=='documento_simulado' for x in res['evidencia'])
        if indice_escena==5:
            assert res['escritura'] is not None and res['escritura']['permitido'] is True
            assert '2025-04-25' not in texto, 'Fecha no documentada encontrada.'
        if indice_escena==6:
            assert res['escritura'] is not None and res['escritura']['permitido'] is False
        if indice_escena==7:
            assert res['caso_id']=='CASO-999' and 'No existen antecedentes suficientes' in texto
        if indice_escena==8:
            assert 'sin registro; no se presume pendiente' in texto
            assert 'pendiente registrado' not in texto
        estado='OK'
        print('Plan ejecutado:', [t['herramienta'] for t in res['ejecutadas']])
        print('Planificador:',res['origen_plan'],'| Ruta final:',res.get('ruta',''))
        print(texto[:2200])
        detalle='Escenario conforme.'
    except Exception as exc:
        estado='ERROR'
        detalle=type(exc).__name__+': '+str(exc)[:360]
        print('VERIFICACIÓN FALLIDA:',detalle)
        res={}
    RESULTADOS.append({'escenario':escena['nombre'],'estado':estado,
        'ruta':res.get('ruta',''),'caso_id':res.get('caso_id') or '',
        'intencion':res.get('intencion',''),'plan_origen':res.get('origen_plan',''),
        'tareas':len(res.get('ejecutadas',[])),
        'plan_detalle':json.dumps(res.get('ejecutadas',[]),ensure_ascii=False),
        'segundos':round(time.perf_counter()-t0,2),'detalle':detalle,
        'respuesta':res.get('respuesta','')})
    if indice_escena < len(ESCENARIOS):
        time.sleep(1)  # Intervalo moderado entre solicitudes de Groq.

reporte_csv=OUTPUT_DRIVE/'registro_operativo_v4.csv'
pd.DataFrame(RESULTADOS).to_csv(reporte_csv,index=False,encoding='utf-8-sig')
print('\nRESULTADO DE PRUEBAS OPERATIVAS')
display(pd.DataFrame(RESULTADOS)[['escenario','estado','plan_origen','tareas','segundos']])
aciertos=sum(x['estado']=='OK' for x in RESULTADOS)
print('Escenarios conformes:',aciertos,'de',len(ESCENARIOS))
assert aciertos==len(ESCENARIOS), (
   'No se cumplen todos los escenarios. Revisar registro_operativo_v4.csv antes de presentar.')


════════════════════════════════════════════════════════════════════
01 · Síntesis integral
════════════════════════════════════════════════════════════════════
Plan ejecutado: ['consultar_registros']
Planificador: groq | Ruta final: respuesta_verificada
ANTECEDENTES DOCUMENTADOS — fuentes de demostración.
Caso: CASO-002
[ORT-2025-CASO-002] Seguimiento 2025: CASO-002. Alta 2025-01-09. Control 2025-01-24. Motivo: Seguimiento administrativo posalta. Gestión 1: pendiente — Confirmar asistencia a control. Gestión 2: realizada — Contacto inicial registrado.
[INT-CASO-002] Interconsulta y traslado: CASO-002. Interconsulta=pendiente. Traslado=pendiente. Días de espera registrados=9. Observación interconsulta=Esperando respuesta de coordinación. Observación traslado=Traslado pendiente de asignación.
Fuentes verificadas: INT-CASO-002, ORT-2025-CASO-002
Control: datos ficticios; confirmación profesional requerida; sin decisiones clínicas.

═══════════════════════════════════════════════════════

,escenario,estado,plan_origen,tareas,segundos
0,01 · Síntesis integral,OK,groq,1,1.77
1,02 · Memoria contextual,OK,groq,1,1.68
2,03 · Planificación adaptativa,OK,groq,8,2.15
3,04 · Consulta documental,OK,groq,1,1.60
4,05 · Escritura autorizada,OK,groq,2,2.40
5,06 · Escritura denegada,OK,groq,1,7.56
6,07 · Caso inexistente,OK,groq,1,4.42
7,08 · Ausencia de datos,OK,groq,1,7.88


Escenarios conformes: 8 de 8


## 12 — Evidencias y comprobaciones de persistencia
Se verifica que **ambas** bases SQLite se copiaron a Drive, que el historial del grafo contiene checkpoints del hilo utilizado y que la memoria puede leerse desde un archivo persistido con una conexión nueva (simulando apertura tras reinicio). Se consultan las ejecuciones **del proyecto concreto** en LangSmith, no simplemente la autenticación general. El estado de trazas se registra como `VERIFICADO`, `NO_VERIFICADO` o `ERROR`; solo `VERIFICADO` completa la demostración técnica.

In [13]:
from langsmith import Client

assert RUTA_APP_DRIVE.exists() and RUTA_GRAFO_DRIVE.exists(), 'Falta sincronizar bases de Drive.'

# Comprobar lectura desde Drive con nuevas conexiones.
with sqlite3.connect(str(RUTA_APP_DRIVE)) as con_prueba:
    total_memoria = con_prueba.execute('SELECT COUNT(*) FROM memorias').fetchone()[0]
    total_borradores = con_prueba.execute('SELECT COUNT(*) FROM borradores').fetchone()[0]
    hilo=con_prueba.execute("SELECT caso_id FROM memorias WHERE hilo='gerencia-operativa-01' AND caso_id IS NOT NULL ORDER BY id DESC LIMIT 1").fetchone()
    assert hilo and hilo[0]=='CASO-002', 'No se recuperó la continuidad de memoria desde Drive.'
with sqlite3.connect(str(RUTA_GRAFO_DRIVE)) as con_prueba:
    total_checkpoints=con_prueba.execute('SELECT COUNT(*) FROM checkpoints').fetchone()[0]
    assert total_checkpoints>0, 'LangGraph no guardó checkpoints persistentes.'

# Consultar las trazas del proyecto específico durante la ventana de esta demostración.
cliente_langsmith=Client()
trazas=[]
error_trazas=None
for intento in range(3):
    try:
        trazas=list(cliente_langsmith.list_runs(
            project_name=os.environ['LANGSMITH_PROJECT'],
            start_time=INICIO_DEMO_UTC, limit=50))
        if trazas:
            break
    except Exception as exc:
        error_trazas=type(exc).__name__+': '+str(exc)[:180]
    if intento<2:
        time.sleep(3)

estado_trazas='VERIFICADO' if trazas else 'ERROR' if error_trazas else 'NO_VERIFICADO'
resumen={
    'producto':'INGER | Centro de gestión inteligente',
    'fecha_utc':datetime.now(timezone.utc).isoformat(),
    'escenarios_total':len(RESULTADOS),
    'escenarios_conformes':sum(r['estado']=='OK' for r in RESULTADOS),
    'planes_groq':sum(r['plan_origen']=='groq' for r in RESULTADOS),
    'memorias_persistidas':total_memoria,
    'checkpoints_persistidos':total_checkpoints,
    'borradores_simulados':total_borradores,
    'modelo_groq':os.environ['GROQ_MODEL'],
    'proyecto_langsmith':os.environ['LANGSMITH_PROJECT'],
    'trazas_langsmith':len(trazas),
    'estado_trazas':estado_trazas,
    'detalle_trazas':error_trazas or '',
    'advertencia':'Datos completamente ficticios. No uso clínico ni validación productiva.'
}
(OUTPUT_DRIVE/'resumen_operativo_v4.json').write_text(
    json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
for clave,valor in resumen.items():
    print(f'{clave}: {valor}')
print('Evidencias en Google Drive:',OUTPUT_DRIVE)
print('Trazas:', 'https://smith.langchain.com/')
assert estado_trazas=='VERIFICADO', (
    'El código ejecutó escenarios, pero LangSmith no confirmó trazas en el proyecto. '
    'Comprobar permisos, nombre del proyecto e ingestión antes de grabar video.')
print('VALIDACIÓN FINAL: 8/8 escenarios + persistencia + trazas verificadas.')

/tmp/ipykernel_1150/3130259053.py:21: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  trazas=list(cliente_langsmith.list_runs(


producto: INGER | Centro de gestión inteligente
fecha_utc: 2026-11-01T17:15:45.622997+00:00
escenarios_total: 8
escenarios_conformes: 8
planes_groq: 8
memorias_persistidas: 15
checkpoints_persistidos: 125
borradores_simulados: 1
modelo_groq: openai/gpt-oss-20b
proyecto_langsmith: INGER-Gestion-Inteligente
trazas_langsmith: 50
estado_trazas: VERIFICADO
detalle_trazas: 
advertencia: Datos completamente ficticios. No uso clínico ni validación productiva.
Evidencias en Google Drive: /content/drive/MyDrive/INGER_Gestion_Inteligente/reportes
Trazas: https://smith.langchain.com/
VALIDACIÓN FINAL: 8/8 escenarios + persistencia + trazas verificadas.


## 13 — Arquitectura y controles de responsabilidad

```text
PROFESIONAL AUTORIZADO (consultas y solicitudes de borrador)
    │
    ▼
LANGGRAPH · PLANIFICADOR CON GROQ (salida JSON estructurada)
    │   └── Revisión del plan y límites de seguridad
    ▼
COLA DE TAREAS (pendientes → ejecutadas, una a una)
    ├── Consultar caso exacto (fuente estructurada)
    ├── Buscar fragmentos FAISS (filtro y umbral)
    └── Aplicar reglas administrativas simuladas
    │
    ▼
EVALUAR EVIDENCIA (replanificar una vez o abstenerse)
    │
    ▼
GROQ · SELECCIÓN DE IDS DE FUENTE (sin texto factual libre)
    │
    ▼
VERIFICADOR DETERMINISTA (solo datos de fuente original)
    │
    ├── Si escritura: comprobar rol + solicitud + evidencia; luego guardar borrador
    │
    ▼
RESPUESTA CON FUENTES Y VALIDACIÓN PROFESIONAL
    │
    ▼
MEMORIA ADMINISTRATIVA SQLITE       CHECKPOINTS LANGGRAPH SQLITE
    │                                │
    └─────────── SNAPSHOTS INDEPENDIENTES A GOOGLE DRIVE ──────────┘

LANGSMITH: TRAZAS DEL GRAFO + EJECUCIONES DEL MODELO
```

**Protecciones:** datos simulados, separación de bases, permisos antes de escritura, ningún estado vacío convertido automáticamente en pendiente, prohibición de decisión clínica, abstención si faltan fuentes y salida factual sin fechas inventadas. El agente utiliza Groq para planificar y seleccionar evidencia; el texto operativo final se forma solo con datos efectivamente presentes en las fuentes. No se utiliza información real del INGER.

**Documentación de tecnologías:** [Groq Structured Outputs](https://console.groq.com/docs/structured-outputs) · [LangGraph SqliteSaver](https://reference.langchain.com/python/langgraph.checkpoint.sqlite/SqliteSaver) · [LangSmith Client](https://langsmith-sdk.readthedocs.io/en/latest/client/langsmith.client.Client.list_runs.html).

**Límites de comprobación:** las verificaciones locales no equivalen a conectividad real con Groq/Google Drive/LangSmith; la validación en línea debe realizarse en Colab con las claves ya guardadas. El entorno de demostración no sustituye una evaluación de seguridad y privacidad institucional.